In [46]:
import pennylane as qml
from pennylane import numpy as np

In [47]:
A = np.random.randint(1,101,size=(4,4)).astype(float)
b = np.random.randint(1,101,size=(4,1)).astype(float)
B=b
a=A
print(A)
print(b)

[[27. 95. 81. 11.]
 [28. 85. 79. 96.]
 [18. 85. 68. 25.]
 [45. 73. 10. 99.]]
[[96.]
 [37.]
 [ 4.]
 [26.]]


In [48]:
b=b/np.linalg.norm(b)
A=A/np.linalg.norm(A,ord=2)
num_qubits = int(np.log2(b.shape[0]))

In [49]:
outer_b = np.outer(b,b.conj().T)
Identity = np.eye(2**num_qubits)
hamiltonian = A.conj().T@(Identity-outer_b)@A
hamiltonian = 0.5 * (hamiltonian + hamiltonian.conj().T)
hamiltonian = qml.Hermitian(hamiltonian,wires=range(num_qubits))

In [50]:
device = qml.device('default.qubit',wires=num_qubits)
def ansatz(weights,wires):
    qml.BasicEntanglerLayers(weights=weights,wires=wires)

In [51]:
@qml.qnode(device)
def cost_function(weights):
    ansatz(weights,range(num_qubits))
    return qml.expval(hamiltonian)

In [52]:
num_layers = 8
weights = np.random.uniform(0,np.pi,(num_layers,num_qubits),requires_grad=True)
print(f'initial weight is {weights}')
optimiser = qml.AdamOptimizer(stepsize=0.1)
for i in range(800):
    weights,cost = optimiser.step_and_cost(cost_function,weights)
print(f'final weights is {weights}')

initial weight is [[0.67839259 2.09181177]
 [2.62918287 0.31101614]
 [2.52412105 0.35773304]
 [1.28588109 0.70354473]
 [1.43668178 0.57612086]
 [1.02850065 1.88750989]
 [1.5332392  2.24441827]
 [1.9983739  0.55954967]]
final weights is [[0.70608898 1.78575529]
 [2.46450254 0.00495966]
 [2.55181744 0.05167656]
 [1.12120077 0.39748825]
 [1.46437817 0.27006438]
 [0.86382033 1.5814534 ]
 [1.5609356  1.93836179]
 [1.83369357 0.25349319]]


In [53]:
@qml.qnode(device)
def final_state(weights):
    ansatz(weights,wires=range(num_qubits))
    return qml.state()

In [54]:
quantum_solution = final_state(weights)
classical_solution = np.linalg.solve(a,B)
classical_solution = classical_solution/np.linalg.norm(classical_solution)
classical_solution = classical_solution.T
overlap = np.abs(np.vdot(quantum_solution, classical_solution))**2

In [55]:
print(f"Final Quantum State:    {np.round(quantum_solution, 4)}")
print(f"Exact Normalized State: {np.round(classical_solution, 4)}")
print(f"(Overlap):     {overlap:.6f}")

Final Quantum State:    [-1.-0.j  0.+0.j  0.+0.j  0.+0.j]
Exact Normalized State: [[ 0.8707 -0.3775  0.2915 -0.1197]]
(Overlap):     0.758195
